# DeBERTa-v3-base 2-Stage Optimized — Kaggle Notebook 版

基於你 Colab 0.876 那份，整合 family filter + logit-space averaging + sqrt sweep + 5-fold ensemble，改寫成 **Kaggle Notebook 原生版本**（不需金鑰、不需 kagglehub、不需下載檔案）。

## ⚠️ 跑之前先做兩件事

### 1. 掛載資料集
右側欄 **Add Input** → 加這兩個：

| 必須的 Dataset | 來源 |
|---|---|
| `MAP - Charting Student Math Misunderstandings`（競賽資料） | 從 Competitions tab 找 |
| `deberta-v3-base`（預訓練模型） | 搜尋 `deberta-v3-base/transformers/default/1` 那種 |

第二個如果沒掛，**Internet 開關打開**也行（會從 HuggingFace 下載）。

### 2. 設定 Accelerator
右側欄 **Accelerator** → **GPU T4 x2** 或 **GPU P100**

## 跑完後怎麼 submit

1. **Save Version**（右上角）→ 選 **Save & Run All (Commit)**，等 notebook 跑完
2. 跑完後點 notebook 的 **Output** tab
3. 找到 `submission.csv`，右上角 **Submit to Competition**

不用另外寫推論程式碼，**訓練+推論都在這一個 notebook 裡完成**。

## 兩種模式（cell 2 切換）

- **快速版**（`TRAIN_FOLDS = [0]`）：~1 小時，預期 LB **0.876 → ~0.89**
- **完整版**（`TRAIN_FOLDS = [0, 1, 2, 3, 4]`）：~4 小時，預期 LB **0.876 → ~0.91**

## 1. Setup

In [34]:
import os, gc, re, random, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from scipy.special import softmax

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
GPU mem: 15.6 GB


## 2. Hyperparameters

⚠️ 改 `TRAIN_FOLDS` 切換快速/完整版：
- `[0]` → 1-fold 快速試水溫（~1 小時）
- `[0, 1, 2, 3, 4]` → 5-fold ensemble（~4 小時，分數較高）

In [35]:
class CFG:
    MODEL_NAME    = "microsoft/deberta-v3-base"
    MAX_LEN       = 256
    BATCH_SIZE    = 16
    EPOCHS        = 3
    LR            = 2e-5
    WEIGHT_DECAY  = 0.01
    N_FOLDS       = 5
    SEED          = 42

    # ⭐ 控制要訓哪幾個 fold
    TRAIN_FOLDS = [0, 1, 2, 3, 4]
    INFER_FOLDS   = None    # None = 跟 TRAIN_FOLDS 一樣

    # 輸出位置（Kaggle 規定 submission.csv 要在 /kaggle/working/）
    OUTPUT_DIR    = "/kaggle/working/checkpoints"
    WORKING_DIR   = "/kaggle/working"

    # Sqrt 指數 sweep
    SQRT_EXPONENTS = [1.0, 0.5, 0.3, 0.7]

if CFG.INFER_FOLDS is None:
    CFG.INFER_FOLDS = CFG.TRAIN_FOLDS

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"訓練設定:")
print(f"  TRAIN_FOLDS = {CFG.TRAIN_FOLDS}  ({'快速版' if len(CFG.TRAIN_FOLDS)==1 else f'{len(CFG.TRAIN_FOLDS)}-fold ensemble'})")
print(f"  INFER_FOLDS = {CFG.INFER_FOLDS}")
print(f"  EPOCHS={CFG.EPOCHS}  BATCH={CFG.BATCH_SIZE}  LR={CFG.LR}")

訓練設定:
  TRAIN_FOLDS = [0, 1, 2, 3, 4]  (5-fold ensemble)
  INFER_FOLDS = [0, 1, 2, 3, 4]
  EPOCHS=3  BATCH=16  LR=2e-05


## 3. 自動找到資料集路徑

Kaggle 的競賽資料 mount 位置有時是 `/kaggle/input/<slug>/`，有時是 `/kaggle/input/competitions/<slug>/`，這裡兩個都試。

In [36]:
# ─── 找競賽資料 ───
COMP_CANDIDATES = [
    "/kaggle/input/map-charting-student-math-misunderstandings",
    "/kaggle/input/competitions/map-charting-student-math-misunderstandings",
]
DATA_PATH = next((p for p in COMP_CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH is not None, (
    "❌ 找不到競賽資料集。請到右側欄 Add Input → Competitions tab → "
    "加入 'MAP - Charting Student Math Misunderstandings'"
)
print(f"✓ 競賽資料: {DATA_PATH}")
print(f"  檔案: {os.listdir(DATA_PATH)}")

CFG.TRAIN_PATH = os.path.join(DATA_PATH, "train.csv")
CFG.TEST_PATH  = os.path.join(DATA_PATH, "test.csv")
assert os.path.exists(CFG.TRAIN_PATH), f"train.csv 不在 {DATA_PATH}"
assert os.path.exists(CFG.TEST_PATH),  f"test.csv 不在 {DATA_PATH}"

# ─── 找預訓練模型（純 offline，自動掃描整個 /kaggle/input/）───
import json as _json

def find_deberta_base():
    """走訪 /kaggle/input/ 找 deberta-v3-base 的 config.json + tokenizer。"""
    if not os.path.exists("/kaggle/input"):
        return None
    
    candidates = []
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        if "config.json" not in filenames:
            continue
        try:
            with open(os.path.join(dirpath, "config.json")) as f:
                cfg = _json.load(f)
        except Exception:
            continue
        # DeBERTa-v3-base 的特徵
        is_deberta_v3 = (
            cfg.get("model_type") == "deberta-v2"          # v3 內部用 v2 架構名
            and cfg.get("hidden_size") == 768              # base = 768
            and cfg.get("num_hidden_layers") == 12         # base = 12 layers
        )
        # 要有 tokenizer 檔案才算完整
        has_tokenizer = any(n in filenames for n in [
            "spm.model", "tokenizer.json", "tokenizer.model"
        ])
        if is_deberta_v3 and has_tokenizer:
            candidates.append(dirpath)
    return candidates[0] if candidates else None


PRETRAINED = find_deberta_base()
if PRETRAINED is None:
    # 印出 /kaggle/input/ 整個結構幫忙 debug
    print("❌ 找不到 deberta-v3-base 預訓練模型。/kaggle/input/ 內容:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count("/")
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth >= 3:
            dirs.clear()
    raise FileNotFoundError(
        "請右側欄 Add Input → Models tab → 搜尋 'microsoft/deberta-v3-base'，"
        "或 Datasets tab → 搜尋 'deberta v3 base'"
    )

print(f"✓ Offline 預訓練模型: {PRETRAINED}")
print(f"  檔案: {sorted(os.listdir(PRETRAINED))}")

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
print(f"✓ Tokenizer loaded")

✓ 競賽資料: /kaggle/input/competitions/map-charting-student-math-misunderstandings
  檔案: ['sample_submission.csv', 'train.csv', 'test.csv']
✓ Offline 預訓練模型: /kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1
  檔案: ['README.md', 'config.json', 'gitattributes', 'pytorch_model.bin', 'rust_model.ot', 'spm.model', 'tf_model.h5', 'tokenizer_config.json']


The tokenizer you are loading from '/kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ Tokenizer loaded


## 4. Data preprocessing（同你 Colab 版）

In [37]:
def clean_text(s):
    if pd.isna(s): return ""
    s = str(s)
    s = re.sub(r'[,!?;\"\'\[\]\{\}]', ' ', s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_text(df):
    q = df["QuestionText"].fillna("").apply(clean_text)
    a = df["MC_Answer"].fillna("").apply(clean_text)
    e = df["StudentExplanation"].fillna("").apply(clean_text)
    df["text"] = q + " " + a + " " + e
    return df

train = pd.read_csv(CFG.TRAIN_PATH)
test  = pd.read_csv(CFG.TEST_PATH)
train = build_text(train)
test  = build_text(test)
print(f"train: {len(train):,} rows  |  test: {len(test):,} rows")

# Stage 1: Category 標籤
cat_encoder = LabelEncoder()
train["cat_label"] = cat_encoder.fit_transform(train["Category"])
CAT_CLASSES = list(cat_encoder.classes_)

# Stage 2: Misconception 標籤
train_mis = train[train["Misconception"].notna() & (train["Misconception"] != "NA")].copy().reset_index(drop=True)
mis_encoder = LabelEncoder()
train_mis["mis_label"] = mis_encoder.fit_transform(train_mis["Misconception"])
MIS_CLASSES = list(mis_encoder.classes_)

print(f"\nCategory 類別 ({len(CAT_CLASSES)}):")
for i, c in enumerate(CAT_CLASSES):
    print(f"  {i}: {c:30s}  ({(train['Category']==c).sum():>6,} rows)")
print(f"\nMisconception 類別: {len(MIS_CLASSES)} (僅用 {len(train_mis):,} 筆訓練)")

train: 36,696 rows  |  test: 3 rows

Category 類別 (6):
  0: False_Correct                   (   227 rows)
  1: False_Misconception             ( 9,457 rows)
  2: False_Neither                   ( 6,542 rows)
  3: True_Correct                    (14,802 rows)
  4: True_Misconception              (   403 rows)
  5: True_Neither                    ( 5,265 rows)

Misconception 類別: 35 (僅用 9,860 筆訓練)


## 5. ⭐ 建立 FAMILY_MAP（含覆蓋率診斷）

**跑完看「Test 覆蓋率」這個數字**：
- ≥ 95% → family filter 安全有效
- 90-95% → 靠 model fallback 處理少量未見的 (Q, A)
- < 90% → 可能 train/test 對不上，需要 debug

In [38]:
def build_family_map(train_df):
    fam = train_df["Category"].str.split("_").str[0] + "_"
    df = pd.DataFrame({
        "QuestionId": train_df["QuestionId"],
        "MC_Answer":  train_df["MC_Answer"],
        "family":     fam,
    })
    fam_map = (df.groupby(["QuestionId", "MC_Answer"])["family"]
                 .agg(lambda s: s.value_counts().idxmax())
                 .to_dict())

    # 檢查模糊 (Q,A) — 同 (Q,A) 同時出現兩個 family
    ambig = []
    for (q, a), group in df.groupby(["QuestionId", "MC_Answer"]):
        counts = group["family"].value_counts()
        if len(counts) > 1 and counts.iloc[1] / counts.iloc[0] > 0.2:
            ambig.append((q, a, dict(counts)))
    return fam_map, ambig


FAMILY_MAP, ambiguous = build_family_map(train)
print(f"✓ FAMILY_MAP: {len(FAMILY_MAP):,} 個 (QuestionId, MC_Answer)")
print(f"  模糊 (Q,A) 數: {len(ambiguous)}  {'⚠ 需要 debug' if len(ambiguous) > 50 else '✓ 可忽略'}")
if 0 < len(ambiguous) <= 10:
    print(f"  範例: {ambiguous[:3]}")

# Test 集覆蓋率
test_seen = sum(1 for q, a in zip(test["QuestionId"], test["MC_Answer"]) if (q, a) in FAMILY_MAP)
coverage = test_seen / len(test)
print(f"\n📊 Test 覆蓋率: {test_seen:,}/{len(test):,} = {100*coverage:.2f}%")

if coverage >= 0.95:
    print("✅ 覆蓋率很高，family filter 安全可用")
elif coverage >= 0.90:
    print("⚠ 覆蓋率偏低，未見 (Q,A) 會用 model fallback")
else:
    print("❌ 覆蓋率太低，可能 train/test (Q,A) 不對齊")

fam_dist = pd.Series(FAMILY_MAP.values()).value_counts()
print(f"\nFAMILY_MAP 分佈:\n{fam_dist.to_string()}")

✓ FAMILY_MAP: 60 個 (QuestionId, MC_Answer)
  模糊 (Q,A) 數: 0  ✓ 可忽略

📊 Test 覆蓋率: 3/3 = 100.00%
✅ 覆蓋率很高，family filter 安全可用

FAMILY_MAP 分佈:
False_    45
True_     15


## 6. Dataset

In [39]:
class MathDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=CFG.MAX_LEN,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

## 7. Training function（支援多 fold）

In [40]:
def train_one_fold(train_df, label_col, num_labels, fold, save_path):
    """訓練單個 fold 並儲存到 save_path。"""
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splits = list(skf.split(train_df, train_df[label_col]))
    train_idx, val_idx = splits[fold]

    tr_df = train_df.iloc[train_idx]
    va_df = train_df.iloc[val_idx]

    train_data = MathDataset(tr_df["text"].values, tr_df[label_col].values)
    val_data   = MathDataset(va_df["text"].values, va_df[label_col].values)

    train_loader = DataLoader(train_data, batch_size=CFG.BATCH_SIZE,   shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=CFG.BATCH_SIZE*2, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=num_labels, torch_dtype=torch.float32,
    ).to(device)
    optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda')

    best_loss = float('inf')
    for epoch in range(CFG.EPOCHS):
        model.train()
        for batch in tqdm(train_loader, desc=f"Fold {fold} ep{epoch+1}", leave=False):
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device),
                )
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation
        model.eval()
        val_loss = 0.0
        n_correct, n_total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(
                        input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["labels"].to(device),
                    )
                    val_loss += outputs.loss.item()
                preds = outputs.logits.argmax(-1)
                n_correct += (preds == batch["labels"].to(device)).sum().item()
                n_total   += batch["labels"].size(0)

        val_loss /= len(val_loader)
        val_acc = n_correct / n_total
        print(f"  Fold {fold} ep{epoch+1}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), save_path)

    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return best_loss

## 8. Train Category model(s)

In [41]:
print("🚀 訓練 Category 模型 (6 分類)...\n")
cat_results = {}
for fold in CFG.TRAIN_FOLDS:
    save_path = f"{CFG.OUTPUT_DIR}/category_fold{fold}.pt"
    print(f"\n──── Category fold {fold} ────")
    val_loss = train_one_fold(train, "cat_label", len(CAT_CLASSES), fold, save_path)
    cat_results[fold] = val_loss
    print(f"✓ fold {fold} 完成 (best val_loss={val_loss:.4f})")

print(f"\n總結 — Category:")
for f, v in cat_results.items():
    print(f"  fold {f}: val_loss = {v:.4f}")

🚀 訓練 Category 模型 (6 分類)...


──── Category fold 0 ────


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias           

Fold 0 ep1:   0%|          | 0/1835 [00:00<?, ?it/s]

  Fold 0 ep1: val_loss=0.7244  val_acc=0.6866


...

  Fold 4 ep1: val_loss=0.5527  val_acc=0.7840


Fold 4 ep2:   0%|          | 0/1835 [00:00<?, ?it/s]

  Fold 4 ep2: val_loss=0.5458  val_acc=0.7814


Fold 4 ep3:   0%|          | 0/1835 [00:00<?, ?it/s]

  Fold 4 ep3: val_loss=0.5513  val_acc=0.8095
✓ fold 4 完成 (best val_loss=0.5458)

總結 — Category:
  fold 0: val_loss = 0.5097
  fold 1: val_loss = 0.4370
  fold 2: val_loss = 1.3858
  fold 3: val_loss = 1.3843
  fold 4: val_loss = 0.5458


## 9. Train Misconception model(s)

In [42]:
print("🚀 訓練 Misconception 模型 (35 分類)...\n")
mis_results = {}
for fold in CFG.TRAIN_FOLDS:
    save_path = f"{CFG.OUTPUT_DIR}/miscon_fold{fold}.pt"
    print(f"\n──── Misconception fold {fold} ────")
    val_loss = train_one_fold(train_mis, "mis_label", len(MIS_CLASSES), fold, save_path)
    mis_results[fold] = val_loss
    print(f"✓ fold {fold} 完成 (best val_loss={val_loss:.4f})")

print(f"\n總結 — Misconception:")
for f, v in mis_results.items():
    print(f"  fold {f}: val_loss = {v:.4f}")

🚀 訓練 Misconception 模型 (35 分類)...


──── Misconception fold 0 ────


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias           

Fold 0 ep1:   0%|          | 0/493 [00:00<?, ?it/s]

  Fold 0 ep1: val_loss=0.4533  val_acc=0.8580


...

  Fold 4 ep1: val_loss=0.5483  val_acc=0.7819


Fold 4 ep2:   0%|          | 0/493 [00:00<?, ?it/s]

  Fold 4 ep2: val_loss=0.2851  val_acc=0.9037


Fold 4 ep3:   0%|          | 0/493 [00:00<?, ?it/s]

  Fold 4 ep3: val_loss=0.2285  val_acc=0.9214
✓ fold 4 完成 (best val_loss=0.2285)

總結 — Misconception:
  fold 0: val_loss = 0.2075
  fold 1: val_loss = 0.1965
  fold 2: val_loss = 0.2565
  fold 3: val_loss = 3.0652
  fold 4: val_loss = 0.2285


## 10. ⭐ Inference: 取 logits（給 logit-space averaging 用）

In [43]:
test_data = MathDataset(test["text"].values)
test_loader = DataLoader(test_data, batch_size=CFG.BATCH_SIZE*2, shuffle=False)

@torch.no_grad()
def get_logits_single(model_path, num_labels):
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=num_labels
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_logits = []
    for batch in tqdm(test_loader, desc=f"Predict {os.path.basename(model_path)}"):
        with torch.amp.autocast('cuda', dtype=torch.float16):
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
        all_logits.append(outputs.logits.float().cpu().numpy())
    del model
    gc.collect(); torch.cuda.empty_cache()
    return np.vstack(all_logits)


def get_logits_ensemble(model_paths, num_labels):
    print(f"Ensembling {len(model_paths)} folds...")
    fold_logits = [get_logits_single(p, num_labels) for p in model_paths]
    mean_logits = np.mean(fold_logits, axis=0)        # logit-space averaging
    probs = softmax(mean_logits, axis=-1)
    return mean_logits, probs


cat_paths = [f"{CFG.OUTPUT_DIR}/category_fold{f}.pt" for f in CFG.INFER_FOLDS]
mis_paths = [f"{CFG.OUTPUT_DIR}/miscon_fold{f}.pt"   for f in CFG.INFER_FOLDS]
for p in cat_paths + mis_paths:
    assert os.path.exists(p), f"找不到 checkpoint: {p}"

print(f"🔮 預測 Category... ({len(cat_paths)} fold)")
cat_logits, cat_probs = get_logits_ensemble(cat_paths, len(CAT_CLASSES))
print(f"   cat_probs shape: {cat_probs.shape}")

print(f"\n🔮 預測 Misconception... ({len(mis_paths)} fold)")
mis_logits, mis_probs = get_logits_ensemble(mis_paths, len(MIS_CLASSES))
print(f"   mis_probs shape: {mis_probs.shape}")

🔮 預測 Category... (5 fold)
Ensembling 5 folds...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias           

Predict category_fold0.pt:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Predict miscon_fold3.pt:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/kevinbnisch/microsoftdeberta-v3-base/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias           

Predict miscon_fold4.pt:   0%|          | 0/1 [00:00<?, ?it/s]

   mis_probs shape: (3, 35)


## 11. ⭐ Family filter（含 model fallback）

In [44]:
TRUE_M_IDX    = CAT_CLASSES.index("True_Misconception")
FALSE_M_IDX   = CAT_CLASSES.index("False_Misconception")
TRUE_CAT_IDX  = [i for i, c in enumerate(CAT_CLASSES) if c.startswith("True_")]
FALSE_CAT_IDX = [i for i, c in enumerate(CAT_CLASSES) if c.startswith("False_")]

def determine_family(qid, mc_ans, cat_prob_row):
    if (qid, mc_ans) in FAMILY_MAP:
        return FAMILY_MAP[(qid, mc_ans)]
    # Model-based fallback
    true_mass  = cat_prob_row[TRUE_CAT_IDX].sum()
    false_mass = cat_prob_row[FALSE_CAT_IDX].sum()
    return "True_" if true_mass >= false_mass else "False_"


# 預計算每一筆 test 的 family
n_lookup, n_fallback = 0, 0
families = []
for b in range(len(test)):
    qid = test.iloc[b]["QuestionId"]
    ans = test.iloc[b]["MC_Answer"]
    if (qid, ans) in FAMILY_MAP:
        families.append(FAMILY_MAP[(qid, ans)])
        n_lookup += 1
    else:
        families.append(determine_family(qid, ans, cat_probs[b]))
        n_fallback += 1

print(f"Family 判定:")
print(f"  FAMILY_MAP 查到: {n_lookup:>6,} ({100*n_lookup/len(test):.1f}%)")
print(f"  Model fallback : {n_fallback:>6,} ({100*n_fallback/len(test):.1f}%)")
print(f"  分佈: {pd.Series(families).value_counts().to_dict()}")

Family 判定:
  FAMILY_MAP 查到:      3 (100.0%)
  Model fallback :      0 (0.0%)
  分佈: {'True_': 2, 'False_': 1}


## 12. ⭐ 產出 submission

| 檔名 | sqrt 指數 | family filter | 說明 |
|---|---|---|---|
| `submission.csv` | 0.5 | ✅ | **預設提交**（你 sqrt0.5 + family filter） |
| `submission_v1_baseline.csv` | 1.0 | ❌ | 你原版基準 |
| `submission_v3_sqrt03.csv` | 0.3 | ✅ | misconception 影響更大 |
| `submission_v4_sqrt07.csv` | 0.7 | ✅ | category 影響更大 |

預設 `submission.csv` 是 v2（最有信心的版本）。如果想 submit 別的，重跑 notebook 時把 cell 13 的 `PRIMARY_VERSION` 改成 v3 或 v4。

In [46]:
def generate_submission(sqrt_exponent: float, use_family_filter: bool, fname: str):
    rows = []
    for b in range(len(test)):
        scores = {}

        # 1. NA 類別
        for j, c_name in enumerate(CAT_CLASSES):
            if c_name not in ("True_Misconception", "False_Misconception"):
                scores[f"{c_name}:NA"] = float(cat_probs[b, j])

        # 2. Misconception 類別: P(Cat)^sqrt × P(Mis)
        p_true_mis  = float(cat_probs[b, TRUE_M_IDX])  ** sqrt_exponent
        p_false_mis = float(cat_probs[b, FALSE_M_IDX]) ** sqrt_exponent
        for k, m_name in enumerate(MIS_CLASSES):
            scores[f"True_Misconception:{m_name}"]  = p_true_mis  * float(mis_probs[b, k])
            scores[f"False_Misconception:{m_name}"] = p_false_mis * float(mis_probs[b, k])

        # 3. Family filter
        if use_family_filter:
            family = families[b]
            scores = {k: v for k, v in scores.items() if k.startswith(family)}
            if len(scores) < 3:
                na_fillers = [f"{c}:NA" for c in CAT_CLASSES if c.startswith(family)]
                for f_label in na_fillers:
                    scores.setdefault(f_label, 0.0)

        # 4. Top-3
        top3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
        rows.append({
            "row_id": test.iloc[b]["row_id"] if "row_id" in test.columns else b,
            "Category:Misconception": " ".join([t[0] for t in top3]),
        })

    sub = pd.DataFrame(rows)
    sub.to_csv(fname, index=False)

    # 驗證
    assert len(sub) == len(test), f"列數錯誤: {len(sub)} vs {len(test)}"
    assert sub["Category:Misconception"].str.split(" ").apply(len).eq(3).all(), "有列預測不是 3 個"
    print(f"  ✓ {fname}  ({len(sub):,} rows)")
    return sub


print("📦 產出 submission...\n")
sub_v1 = generate_submission(1.0, False, f"{CFG.WORKING_DIR}/submission_v1_baseline.csv")
sub_v2 = generate_submission(0.5, True,  f"{CFG.WORKING_DIR}/submission_v2_sqrt05.csv")
sub_v3 = generate_submission(0.3, True,  f"{CFG.WORKING_DIR}/submission_v3_sqrt03.csv")
sub_v4 = generate_submission(0.7, True,  f"{CFG.WORKING_DIR}/submission_v4_sqrt07.csv")
print()

📦 產出 submission...

  ✓ /kaggle/working/submission_v1_baseline.csv  (3 rows)
  ✓ /kaggle/working/submission_v2_sqrt05.csv  (3 rows)
  ✓ /kaggle/working/submission_v3_sqrt03.csv  (3 rows)
  ✓ /kaggle/working/submission_v4_sqrt07.csv  (3 rows)



## 13. 把預設提交版本複製成 `submission.csv`

In [47]:
# Kaggle 競賽預設找 /kaggle/working/submission.csv
# 想換策略就改這個變數重跑：
PRIMARY_VERSION = "v2"    # 可選: v1, v2, v3, v4

version_map = {
    "v1": "submission_v1_baseline.csv",
    "v2": "submission_v2_sqrt05.csv",
    "v3": "submission_v3_sqrt03.csv",
    "v4": "submission_v4_sqrt07.csv",
}

primary_src = f"{CFG.WORKING_DIR}/{version_map[PRIMARY_VERSION]}"
primary_dst = f"{CFG.WORKING_DIR}/submission.csv"
shutil.copy(primary_src, primary_dst)

print(f"✓ 預設提交版本: {PRIMARY_VERSION} ({version_map[PRIMARY_VERSION]})")
print(f"✓ 複製到 {primary_dst}")
print(f"\n所有產出檔案:")
for f in sorted(os.listdir(CFG.WORKING_DIR)):
    full = f"{CFG.WORKING_DIR}/{f}"
    if os.path.isfile(full):
        size_kb = os.path.getsize(full) / 1024
        print(f"  {f:40s}  ({size_kb:>8.1f} KB)")

✓ 預設提交版本: v2 (submission_v2_sqrt05.csv)
✓ 複製到 /kaggle/working/submission.csv

所有產出檔案:
  submission.csv                            (     0.2 KB)
  submission_v1_baseline.csv                (     0.2 KB)
  submission_v2_sqrt05.csv                  (     0.2 KB)
  submission_v3_sqrt03.csv                  (     0.2 KB)
  submission_v4_sqrt07.csv                  (     0.2 KB)


## 14. Sanity check + 比較 4 版本的差異

In [48]:
# 各版本 top-1 標籤分佈
print("Top-1 label 分佈（前 6 名）:\n")
for name, sub in [("v1 baseline", sub_v1), ("v2 sqrt0.5+fam", sub_v2),
                  ("v3 sqrt0.3+fam", sub_v3), ("v4 sqrt0.7+fam", sub_v4)]:
    top1 = sub["Category:Misconception"].str.split(" ").str[0]
    print(f"── {name} ──")
    print(top1.value_counts().head(6).to_string())
    print()

# 一致率
def agreement_rate(s1, s2):
    t1 = s1["Category:Misconception"].str.split(" ").str[0]
    t2 = s2["Category:Misconception"].str.split(" ").str[0]
    return (t1 == t2).mean()

print("Top-1 一致率:")
print(f"  v1 ↔ v2 = {100*agreement_rate(sub_v1, sub_v2):.1f}%  (family filter 改變了多少預測)")
print(f"  v2 ↔ v3 = {100*agreement_rate(sub_v2, sub_v3):.1f}%  (sqrt 指數影響)")
print(f"  v2 ↔ v4 = {100*agreement_rate(sub_v2, sub_v4):.1f}%  (sqrt 指數另一個方向)")

Top-1 label 分佈（前 6 名）:

── v1 baseline ──
Category:Misconception
True_Correct:NA            1
False_Misconception:WNB    1
True_Neither:NA            1

── v2 sqrt0.5+fam ──
Category:Misconception
True_Correct:NA            1
False_Misconception:WNB    1
True_Neither:NA            1

── v3 sqrt0.3+fam ──
Category:Misconception
True_Correct:NA            1
False_Misconception:WNB    1
True_Neither:NA            1

── v4 sqrt0.7+fam ──
Category:Misconception
True_Correct:NA            1
False_Misconception:WNB    1
True_Neither:NA            1

Top-1 一致率:
  v1 ↔ v2 = 100.0%  (family filter 改變了多少預測)
  v2 ↔ v3 = 100.0%  (sqrt 指數影響)
  v2 ↔ v4 = 100.0%  (sqrt 指數另一個方向)


---

## 跑完後怎麼 submit

### Step 1：Save Version
右上角 **Save Version** → 選 **Save & Run All (Commit)**

### Step 2：等 notebook 跑完
- 1-fold 快速版：~1 小時
- 5-fold 完整版：~4 小時

### Step 3：Submit 到競賽
- 跑完後到 notebook 的 **Output** tab
- 找到 `submission.csv`（這是預設的 v2）
- 點 **Submit to Competition** → 確認 → submit
- 等 Kaggle 跑完 scoring，看 public LB 分數

### Step 4：A/B test 其他版本（可選）

Kaggle 一天 5 個 submit 額度。想試其他策略：

1. 改 cell 13 的 `PRIMARY_VERSION = "v3"`（或 v4 / v1）
2. **Save Version** 重跑（**注意：模型已經訓好了，但 Kaggle 還是會從頭跑** — 想省時間可以註解掉 cell 8、9 的訓練 cells，前提是上次的 `.pt` 還在 working 目錄）
3. 重新 submit

更省事的做法：**第一次 submit 完成後，把 4 個 CSV 都從 Output 下載下來，到競賽頁面**「Submit Predictions」**直接上傳 CSV submit**（不用重跑 notebook）。

## 不需要分開的「推論 notebook」

整份 pipeline（訓練 → 推論 → 產 submission）都在這個 notebook 裡。`.pt` 檔案只是中間產物，submit 時 Kaggle 用的是 `submission.csv`。

如果你**真的**想分開（例如想反覆 A/B 不同推論策略不用重訓），步驟是：

1. 把這個 notebook 跑完一次，產出 `.pt` 檔
2. 點 notebook **Output** → **New Dataset** → 命名（例如 `my-deberta-folds`）
3. 開新的「inference-only notebook」，掛上這個 dataset，**只跑 cell 1-6 + 10-14**（跳過訓練）

但這對 1-fold 來說沒必要，5-fold 才考慮。